# The Complex Logarithm and Branch Cuts

The real logarithm undoes the real exponential. The complex logarithm should undo the complex exponential, but $e^z$ has period $2\pi i$, so every nonzero number has **infinitely many** logarithms, one for each turn around the origin. This notebook shows how to tame that: pick one value (the **principal logarithm** $\mathrm{Log}$), accept that it must jump somewhere (the **branch cut**), and understand the price — familiar rules like $\log(zw) = \log z + \log w$ only hold up to multiples of $2\pi i$.

We also look at what the computer does on the cut (the sign of a floating-point zero decides the answer) and at the **Riemann surface**, the spiral staircase on which the logarithm becomes single-valued again.

This is notebook 03 of module 03 (**Complex functions**) in the **Learning Complex Numbers** series.

## 1. Introduction

Every positive real number $x$ has exactly one real logarithm, $\ln x$. For complex numbers the question "which $w$ satisfies $e^w = z$?" has the answer

$$
w = \ln|z| + i(\theta + 2\pi k), \qquad k \in \mathbb{Z},
$$

where $\theta$ is any argument of $z$: the modulus pins down the real part, but the angle is only known up to full turns. Negative numbers, which have no real logarithm at all, now have infinitely many complex ones: $\log(-1) = \ldots, -\pi i, \pi i, 3\pi i, \ldots$

This matters well beyond pure mathematics:

- Every library (`cmath`, numpy, MATLAB, C, Fortran) has to choose one value, and they all choose the same principal branch. Knowing where it jumps prevents subtle bugs in phase calculations.
- Complex powers $z^w = e^{w\log z}$ (next notebook), square roots and inverse trig functions inherit the same branch cut.
- Phase unwrapping in signal processing, and winding numbers in contour integration (module 06), are the continuous-logarithm idea in disguise.

**Prerequisites**

- The real natural logarithm and its rules — [Derivative of logarithmic functions](../../../LearningCalculus/hypersklill/02_derivaties/05_derivative_of_logarithmic_functions.ipynb) (LearningCalculus).
- Argument and principal argument $\mathrm{Arg}\, z \in (-\pi, \pi]$, `atan2` — [Polar form, modulus and argument](../01_polar_form_and_euler/01_polar_form_modulus_and_argument.ipynb).
- $e^z$, its period $2\pi i$ and the solutions of $e^z = w$ — [The complex exponential](02_the_complex_exponential.ipynb).
- Mapping pictures and domain colouring — [Complex functions as maps](01_complex_functions_as_maps.ipynb).

After completing this notebook you will be able to:

- List all values of $\log z$ and compute the principal value $\mathrm{Log}\, z$.
- Explain why $\mathrm{Log}$ is discontinuous across the negative real axis, and predict what `cmath.log` returns there (including the signed-zero cases).
- Give counterexamples to $\mathrm{Log}(zw) = \mathrm{Log}\, z + \mathrm{Log}\, w$ and state exactly when it holds.
- Define other branches of the logarithm and describe the Riemann surface of $\log$.

## 2. Theory

### 2.1 The multi-valued logarithm

For $z \neq 0$ the set of all logarithms of $z$ is

$$
\log z = \{\, w \in \mathbb{C} : e^w = z \,\} = \ln|z| + i \arg z = \big\{\ln|z| + i(\mathrm{Arg}\, z + 2\pi k) : k \in \mathbb{Z}\big\}.
$$

Here $\ln$ is the ordinary real logarithm and $\arg z$ is the multi-valued argument. Geometrically the values form a **vertical column** in the $w$-plane, spaced $2\pi$ apart, at height $\mathrm{Re} = \ln|z|$. $\log 0$ is undefined, because $e^w \neq 0$.

### 2.2 The principal logarithm

Choosing the principal argument gives a genuine function:

$$
\boxed{\ \mathrm{Log}\, z = \ln|z| + i\,\mathrm{Arg}\, z, \qquad -\pi < \mathrm{Arg}\, z \le \pi.\ }
$$

Its values fill the horizontal strip $-\pi < \mathrm{Im}\, w \le \pi$, which is exactly the strip on which $e^w$ is one-to-one (notebook 02, Exercise 9). Consequently

$$
e^{\mathrm{Log}\, z} = z \ \text{ for all } z \neq 0, \qquad \text{but} \qquad \mathrm{Log}(e^w) = w \ \text{ only if } -\pi < \mathrm{Im}\, w \le \pi.
$$

For example $\mathrm{Log}(e^{3\pi i}) = \mathrm{Log}(-1) = \pi i \neq 3\pi i$. On the positive real axis $\mathrm{Log}\, x = \ln x$, so $\mathrm{Log}$ extends the real logarithm.

### 2.3 The branch cut

Approach a negative number $-r$ ($r > 0$) from above and from below:

$$
\lim_{\varepsilon \to 0^+} \mathrm{Log}(-r + i\varepsilon) = \ln r + i\pi, \qquad
\lim_{\varepsilon \to 0^+} \mathrm{Log}(-r - i\varepsilon) = \ln r - i\pi.
$$

The imaginary part jumps by $2\pi$. So $\mathrm{Log}$ is **discontinuous along the negative real axis** $(-\infty, 0]$, which is called its **branch cut**. Off the cut, on $\mathbb{C} \setminus (-\infty, 0]$, $\mathrm{Log}$ is continuous and even differentiable, with

$$
\frac{d}{dz} \mathrm{Log}\, z = \frac{1}{z}.
$$

The cut is not a defect of our choice: **no** continuous logarithm exists on any region that contains a loop around $0$. Follow any continuous choice of $\log z$ once around the circle $|z| = 1$: the argument increases by $2\pi$, so the value comes back increased by $2\pi i$, not to where it started. The point $0$, around which this happens, is a **branch point**.

### 2.4 Other branches

Any ray from $0$ can serve as the cut. For a fixed angle $\alpha$, the branch

$$
\log_\alpha z = \ln|z| + i\,\theta, \qquad \theta \in (\alpha, \alpha + 2\pi], \ \ \theta \text{ an argument of } z
$$

is continuous everywhere except on the ray at angle $\alpha$. The principal branch is $\alpha = -\pi$. The branch with $\theta \in [0, 2\pi)$ (cut along the positive real axis) is common in physics. Different branches differ by a constant multiple of $2\pi i$ on each region where both are continuous.

### 2.5 Floating point: the sign of zero picks the side

Numbers stored exactly on the cut, like $-1$, need a convention. IEEE floating point has two zeros, $+0.0$ and $-0.0$, and the C99/Python conventions use the sign of a zero imaginary part to say "approached from above" or "from below":

$$
\texttt{cmath.log(complex(-1, 0.0))} = \pi i, \qquad \texttt{cmath.log(complex(-1, -0.0))} = -\pi i.
$$

Watch out: the literal `-1-0j` is computed as `(-1) - (0j)`, i.e. `complex(-1 - 0, 0 - 0)`, and $0 - 0 = +0.0$, so it lands on the **upper** side. Use `complex(-1, -0.0)` to get the lower side.

### 2.6 Which logarithm laws survive?

As **sets** of values, the laws are true:

$$
\log(zw) = \log z + \log w, \qquad \log(1/z) = -\log z.
$$

For the **principal** value they can fail by $2\pi i$. Since $\mathrm{Arg}\, z + \mathrm{Arg}\, w \in (-2\pi, 2\pi]$ but $\mathrm{Arg}(zw)$ must lie in $(-\pi, \pi]$,

$$
\mathrm{Log}(zw) = \mathrm{Log}\, z + \mathrm{Log}\, w + 2\pi i\, k, \qquad k \in \{-1, 0, 1\},
$$

with $k = 0$ exactly when $\mathrm{Arg}\, z + \mathrm{Arg}\, w \in (-\pi, \pi]$. Counterexample: $z = w = -1$ gives $\mathrm{Log}(1) = 0$ but $\mathrm{Log}(-1) + \mathrm{Log}(-1) = 2\pi i$. In particular $\mathrm{Log}(z^2) \neq 2\,\mathrm{Log}\, z$ whenever $\mathrm{Arg}\, z \notin (-\pi/2, \pi/2]$.

The law $\ln(xy) = \ln x + \ln y$ that you know for **positive reals** is the case $\mathrm{Arg} = 0$, where nothing can go wrong.

### 2.7 The Riemann surface

Here is a way to make $\log$ single-valued without cutting anything away. Take infinitely many copies ("sheets") of the plane, each cut along the negative real axis. On sheet $k$ use the values $\ln|z| + i(\mathrm{Arg}\, z + 2\pi k)$. Now glue the upper edge of the cut on sheet $k$ to the lower edge on sheet $k + 1$. Walking counter-clockwise around $0$, you never hit a cut: you climb smoothly onto the next sheet, and $\mathrm{Im}\log$ keeps increasing.

The result is an infinite **spiral staircase** (a helicoid) over the punctured plane, called the **Riemann surface of $\log$**. On it, $\log$ is a continuous single-valued function. The branch cut is just the artefact of flattening the staircase back down to one sheet. We plot it in §5.4.

## 3. Worked Examples

### Example 1 — principal values

| $z$ | $\lvert z\rvert$ | $\mathrm{Arg}\, z$ | $\mathrm{Log}\, z$ |
|---|---|---|---|
| $e$ | $e$ | $0$ | $1$ |
| $-1$ | $1$ | $\pi$ | $i\pi$ |
| $i$ | $1$ | $\pi/2$ | $i\pi/2$ |
| $1 - i$ | $\sqrt2$ | $-\pi/4$ | $\frac12\ln 2 - i\frac\pi4$ |
| $-3$ | $3$ | $\pi$ | $\ln 3 + i\pi$ |

Check one: $e^{\frac12\ln2 - i\pi/4} = \sqrt2\,\big(\cos(-\tfrac\pi4) + i\sin(-\tfrac\pi4)\big) = \sqrt2\left(\tfrac{\sqrt2}{2} - \tfrac{\sqrt2}{2}i\right) = 1 - i$ ✓.

### Example 2 — all values of $\log(-1)$ and $\log(i)$

$$
\log(-1) = i\pi(2k + 1) = \{\ldots, -3\pi i, -\pi i, \pi i, 3\pi i, \ldots\}, \qquad
\log i = i\left(\tfrac\pi2 + 2\pi k\right) = \{\ldots, -\tfrac{3\pi}{2} i, \tfrac\pi2 i, \tfrac{5\pi}{2} i, \ldots\}.
$$

Both sets lie on the imaginary axis because $\ln|{-1}| = \ln|i| = 0$.

### Example 3 — the jump across the cut

Take $z_\pm = -4 \pm 0.001i$, just above and below the cut. Both have modulus $\approx 4$ and $\mathrm{Arg}\, z_\pm \approx \pm(\pi - 0.00025)$. So

$$
\mathrm{Log}\, z_+ \approx 1.3863 + 3.1413i, \qquad \mathrm{Log}\, z_- \approx 1.3863 - 3.1413i.
$$

Two points $0.002$ apart have logarithms almost $2\pi$ apart.

### Example 4 — $\mathrm{Log}(zw) \neq \mathrm{Log}\, z + \mathrm{Log}\, w$

Take $z = w = -1 + i$, with $\mathrm{Arg} = 3\pi/4$ each. Then $zw = (-1 + i)^2 = -2i$, with $\mathrm{Arg}(-2i) = -\pi/2$:

$$
\mathrm{Log}(zw) = \ln 2 - i\tfrac{\pi}{2}, \qquad
\mathrm{Log}\, z + \mathrm{Log}\, w = 2\left(\tfrac12\ln 2 + i\tfrac{3\pi}{4}\right) = \ln 2 + i\tfrac{3\pi}{2}.
$$

The difference is exactly $2\pi i$ ($k = -1$ in §2.6): the sum of arguments $3\pi/2$ overshot $\pi$ and had to be wrapped back.

### Example 5 — $\mathrm{Log}(e^w)$ does not always return $w$

$w = 2 + 5i$: $e^w = e^2 e^{5i}$, and the principal argument of $e^{5i}$ is $5 - 2\pi \approx -1.283$. Hence

$$
\mathrm{Log}(e^{2 + 5i}) = 2 + (5 - 2\pi)i \approx 2 - 1.283i.
$$

$\mathrm{Log}$ always brings the imaginary part back into $(-\pi, \pi]$.

### Example 6 — a branch with a different cut

With $\theta \in [0, 2\pi)$ (cut along the positive real axis): $\log(-i) = i\frac{3\pi}{2}$, whereas $\mathrm{Log}(-i) = -i\frac{\pi}{2}$. They differ by $2\pi i$. The new branch is continuous across the negative real axis: just above $-1$ it is $\approx i\pi$, just below it is also $\approx i\pi$.

## 4. Python Examples

### 4.1 `cmath.log` is the principal logarithm

In [ ]:
import cmath
import math

for z in [math.e, -1, 1j, 1 - 1j, -3]:
    w = cmath.log(z)
    principal_by_hand = complex(math.log(abs(z)), cmath.phase(z))
    print(f"Log({z}) = {w:.6f}   by hand: {principal_by_hand:.6f}   e^Log z = {cmath.exp(w):.6f}")

### 4.2 All the values: a column spaced $2\pi$ apart

In [ ]:
import cmath
import math


def all_logs(z, ks=range(-2, 3)):
    principal = cmath.log(z)
    return [principal + 2j * math.pi * k for k in ks]


for z in [-1, 1j, 2]:
    values = all_logs(z)
    print(f"log({z}):", ", ".join(f"{v.real:.3f}{v.imag:+.3f}i" for v in values))
    print("   all exponentiate back:", all(abs(cmath.exp(v) - z) < 1e-12 for v in values))

### 4.3 The jump across the cut and the signed zero

In [ ]:
import cmath

for epsilon in [1e-1, 1e-3, 1e-6]:
    above = cmath.log(complex(-4, epsilon))
    below = cmath.log(complex(-4, -epsilon))
    print(f"eps = {epsilon:.0e}:  Log(-4 + i eps) = {above:.6f}   Log(-4 - i eps) = {below:.6f}   "
          f"jump in Im = {above.imag - below.imag:.6f}")

print()
print("complex(-1,  0.0) ->", cmath.log(complex(-1, 0.0)))
print("complex(-1, -0.0) ->", cmath.log(complex(-1, -0.0)))
print("the literal -1-0j is", repr(-1 - 0j), "-> Log =", cmath.log(-1 - 0j), "  (upper side!)")
print("cmath.phase(complex(-1, -0.0)) =", cmath.phase(complex(-1, -0.0)))

numpy follows the same principal branch and the same signed-zero rule:

In [ ]:
import numpy as np

z = np.array([-1 + 0j, complex(-1, -0.0), 1j, -1j, -2 + 1e-12j, -2 - 1e-12j])
print("z      :", z)
print("np.log :", np.log(z))

### 4.4 How often does $\mathrm{Log}(zw) = \mathrm{Log}\, z + \mathrm{Log}\, w$ fail?

Sample random pairs and compute the correction $k = \big(\mathrm{Log}(zw) - \mathrm{Log}\, z - \mathrm{Log}\, w\big) / 2\pi i$. It should always be an integer in $\{-1, 0, 1\}$. How often is it nonzero?

In [ ]:
import numpy as np

rng = np.random.default_rng(seed=0)
n = 100_000
z = rng.uniform(0.2, 3, n) * np.exp(1j * rng.uniform(-np.pi, np.pi, n))
w = rng.uniform(0.2, 3, n) * np.exp(1j * rng.uniform(-np.pi, np.pi, n))

k = (np.log(z * w) - np.log(z) - np.log(w)) / (2j * np.pi)
print("k is (numerically) an integer:", np.allclose(k.imag, 0) and np.allclose(k.real, np.round(k.real)))
values, counts = np.unique(np.round(k.real).astype(int), return_counts=True)
for value, count in zip(values, counts):
    print(f"k = {value:+d}: {count / n:.3f} of pairs")

predicted_k0 = np.abs(np.angle(z) + np.angle(w)) < np.pi
print("k = 0 exactly when |Arg z + Arg w| < pi:", np.array_equal(np.round(k.real) == 0, predicted_k0))

z0 = w0 = -1 + 1j
print("Example 4: Log(zw) =", np.log(z0 * w0), "  Log z + Log w =", np.log(z0) + np.log(w0))

For uniformly random arguments the sum $\mathrm{Arg}\, z + \mathrm{Arg}\, w$ has a triangular distribution on $(-2\pi, 2\pi]$, which puts probability $\frac18$ above $\pi$ and $\frac18$ below $-\pi$. So the "obvious" law fails for a quarter of all pairs — far too often to ignore.

### 4.5 sympy refuses to apply an invalid law

sympy only expands $\log(xy)$ when it knows the law is safe (for example when the symbols are positive).

In [ ]:
import sympy as sp

z, w = sp.symbols("z w")                    # arbitrary complex
p, q = sp.symbols("p q", positive=True)     # positive reals

print("expand_log(log(z*w))       :", sp.expand_log(sp.log(z * w)))
print("expand_log(log(p*q))       :", sp.expand_log(sp.log(p * q)))
print("log(-1)                    :", sp.log(-1))
print("log(I), log(1 - I)         :", sp.log(sp.I), ",", sp.simplify(sp.log(1 - sp.I)))
print("log(exp(3*pi*I))           :", sp.log(sp.exp(3 * sp.pi * sp.I)))
print("log(-1) + log(-1) - log(1) :", sp.log(-1) + sp.log(-1) - sp.log(1))

### 4.6 A logarithm with any cut you like

`log_branch(z, alpha)` returns the logarithm whose imaginary part lies in $(\alpha, \alpha + 2\pi]$. With $\alpha = -\pi$ it reproduces `cmath.log`.

In [ ]:
import cmath
import math


def log_branch(z, alpha):
    principal = cmath.log(z)
    # shift Im into (alpha, alpha + 2 pi] by a whole number of turns
    turns = math.ceil((alpha - principal.imag) / (2 * math.pi))
    candidate = principal + 2j * math.pi * turns
    if candidate.imag <= alpha:
        candidate += 2j * math.pi
    return candidate


for z in [-1j, 1 - 1j, -1 + 1e-9j, -1 - 1e-9j]:
    print(f"z = {z}:  principal {log_branch(z, -math.pi):.4f}   [0, 2pi)-type branch {log_branch(z, -1e-15):.4f}")

The last two rows show the point of a different cut: the branch with arguments in $[0, 2\pi)$ gives nearly equal values just above and just below $-1$, while the principal branch jumps.

## 5. Visualizations

### 5.1 The argument as a height map, and the jump

Left: the principal argument $\mathrm{Arg}\, z = \mathrm{Im}\,\mathrm{Log}\, z$ coloured over the plane; the colour jumps from yellow ($+\pi$) to dark ($-\pi$) along the negative real axis. Right: $\mathrm{Im}$ of the logarithm along the circle $z = e^{it}$, $0 \le t \le 4\pi$. The principal value saw-tooths; the **continuous** choice (`np.unwrap`) keeps rising, gaining $2\pi$ per lap.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, (ax_map, ax_path) = plt.subplots(1, 2, figsize=(14, 6))

x = np.linspace(-2, 2, 500)
X, Y = np.meshgrid(x, x)
image = ax_map.imshow(np.angle(X + 1j * Y), origin="lower", extent=[-2, 2, -2, 2], cmap="viridis")
fig.colorbar(image, ax=ax_map, label="Arg z = Im Log z")
ax_map.plot([-2, 0], [0, 0], color="red", linewidth=3, label="branch cut")
ax_map.axhline(0, color="black", linewidth=0.5)
ax_map.axvline(0, color="black", linewidth=0.5)
ax_map.set_aspect("equal")
ax_map.set_xlabel("Re")
ax_map.set_ylabel("Im")
ax_map.set_title("Principal argument: a jump of $2\\pi$ across $(-\\infty, 0]$")
ax_map.legend(loc="upper right")

t = np.linspace(0, 4 * np.pi, 1000)
circle = np.exp(1j * t)
ax_path.plot(t, np.angle(circle), label="Im Log z (principal): jumps")
ax_path.plot(t, np.unwrap(np.angle(circle)), "--", label="continuous log along the path")
for crossing in [np.pi, 3 * np.pi]:
    ax_path.axvline(crossing, color="red", linestyle=":", linewidth=1)
ax_path.set_xticks(np.pi * np.arange(5))
ax_path.set_xticklabels(["0", "$\\pi$", "$2\\pi$", "$3\\pi$", "$4\\pi$"])
ax_path.set_xlabel("$t$   ($z = e^{it}$; red lines: crossing the negative real axis)")
ax_path.set_ylabel("imaginary part of the logarithm")
ax_path.set_title("Twice around the unit circle")
ax_path.grid(True, alpha=0.3)
ax_path.legend()
fig.tight_layout()
plt.show()

### 5.2 $\mathrm{Log}$ as a map: polar grid to rectangular grid

$\mathrm{Log}$ undoes notebook 02's picture. Circles $|z| = r$ become vertical segments $\mathrm{Re}\, w = \ln r$; rays $\mathrm{Arg}\, z = \theta$ become horizontal lines $\mathrm{Im}\, w = \theta$. The cut plane is opened up into the strip $-\pi < \mathrm{Im}\, w \le \pi$. The two rays hugging the cut from above (red) and below (purple) are neighbours in the $z$-plane but land on **opposite** edges of the strip.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, (ax_z, ax_w) = plt.subplots(1, 2, figsize=(13, 6), gridspec_kw={"width_ratios": [1, 1.2]})
radii = np.linspace(0.02, 3, 300)
angles = np.linspace(-np.pi + 1e-3, np.pi, 600)

for r in [0.25, 0.5, 1, 2, 3]:
    circle = r * np.exp(1j * angles)
    ax_z.plot(circle.real, circle.imag, color="tab:blue", linewidth=1)
    ax_w.plot(np.log(circle).real, np.log(circle).imag, color="tab:blue", linewidth=1)
for theta in np.linspace(-3 * np.pi / 4, np.pi / 2, 6):
    ray = radii * np.exp(1j * theta)
    ax_z.plot(ray.real, ray.imag, color="tab:orange", linewidth=1)
    ax_w.plot(np.log(ray).real, np.log(ray).imag, color="tab:orange", linewidth=1)
for theta, colour, label in [(np.pi - 0.05, "red", "just above the cut"), (-np.pi + 0.05, "purple", "just below the cut")]:
    ray = radii * np.exp(1j * theta)
    ax_z.plot(ray.real, ray.imag, color=colour, linewidth=2.5, label=label)
    ax_w.plot(np.log(ray).real, np.log(ray).imag, color=colour, linewidth=2.5)

ax_z.plot([-3.2, 0], [0, 0], color="black", linewidth=4, alpha=0.3)
for level in (-np.pi, np.pi):
    ax_w.axhline(level, color="gray", linestyle="--", linewidth=1)
ax_z.set_title("z-plane: circles and rays (cut shaded)")
ax_w.set_title("w = Log z: the strip $-\\pi < \\mathrm{Im}\\, w \\leq \\pi$")
ax_w.set_yticks(np.pi * np.array([-1, -0.5, 0, 0.5, 1]))
ax_w.set_yticklabels(["$-\\pi$", "$-\\pi/2$", "0", "$\\pi/2$", "$\\pi$"])
ax_z.legend(loc="upper left", fontsize=9)
for ax in (ax_z, ax_w):
    ax.axhline(0, color="black", linewidth=0.8)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_aspect("equal")
    ax.set_xlabel("Re")
    ax.set_ylabel("Im")
    ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

### 5.3 Domain colouring of $\mathrm{Log}\, z$

The seam along the negative real axis is the branch cut: above it the output has imaginary part near $+\pi$ (its colour reflects an argument near $+\pi/2$), below it near $-\pi$ (argument near $-\pi/2$). There is a black dot at the zero $z = 1$ and white near the branch point $0$, where $|\mathrm{Log}\, z| \to \infty$ — but unlike a pole, the colours around $0$ do not form a clean cycle, because the function is not even continuous there.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import hsv_to_rgb


def domain_colouring(f, re_range, im_range, resolution=600):
    X, Y = np.meshgrid(np.linspace(*re_range, resolution), np.linspace(*im_range, resolution))
    W = f(X + 1j * Y)
    hue = (np.angle(W) / (2 * np.pi)) % 1.0
    m = (2 / np.pi) * np.arctan(np.abs(W))
    hsv = np.stack([hue, np.clip(2 - 2 * m, 0, 1), np.clip(2 * m, 0, 1)], axis=-1)
    return hsv_to_rgb(hsv)


fig, axes = plt.subplots(1, 2, figsize=(13, 6))
for ax, f, title in [(axes[0], np.log, "principal Log z: seam on the negative real axis"),
                     (axes[1], lambda z: np.log(-z) + 1j * np.pi, "branch with cut on the positive real axis")]:
    ax.imshow(domain_colouring(f, (-3, 3), (-3, 3)), origin="lower", extent=[-3, 3, -3, 3])
    ax.set_aspect("equal")
    ax.set_xlabel("Re")
    ax.set_ylabel("Im")
    ax.set_title(title)
fig.tight_layout()
plt.show()

The right-hand picture uses $\log z = \mathrm{Log}(-z) + i\pi$, whose imaginary part runs over $(0, 2\pi]$: the seam has moved to the positive real axis, and the negative real axis is now perfectly smooth. Which ray carries the cut is a choice; that **some** ray must carry it is not.

### 5.4 The Riemann surface: a spiral staircase

Plot the point $(\mathrm{Re}\, z, \mathrm{Im}\, z)$ at height $\arg z$, letting the argument run continuously over three turns. Each turn is one sheet. The principal branch is the single turn with height in $(-\pi, \pi]$ (the middle teal turn); flattening it onto the plane is what creates the cut.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

r = np.linspace(0.1, 2, 25)
theta = np.linspace(-3 * np.pi, 3 * np.pi, 300)
R, Theta = np.meshgrid(r, theta)
X = R * np.cos(Theta)
Y = R * np.sin(Theta)

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(projection="3d")
ax.plot_surface(X, Y, Theta, cmap="viridis", alpha=0.8, linewidth=0, rstride=2, cstride=2)
ax.set_xlabel("Re z")
ax.set_ylabel("Im z")
ax.set_zlabel("Im log z")
ax.set_zticks(np.pi * np.arange(-3, 4, 1))
ax.set_zticklabels(["$-3\\pi$", "$-2\\pi$", "$-\\pi$", "0", "$\\pi$", "$2\\pi$", "$3\\pi$"])
ax.set_title("Riemann surface of log z: one turn of the staircase per sheet")
ax.view_init(elev=22, azim=-60)
ax.set_box_aspect(None, zoom=0.85)
plt.show()

### 5.5 All values of $\log z$ at once

For a few points $z$, plot the column of values $\ln|z| + i(\mathrm{Arg}\, z + 2\pi k)$. The principal value (filled marker) is the one inside the shaded strip.

In [ ]:
import cmath
import math

import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(10, 8))
ax.axhspan(-np.pi, np.pi, color="gold", alpha=0.2, label="principal strip $-\\pi < \\mathrm{Im} \\leq \\pi$")
for z, colour in [(-1, "tab:red"), (1j, "tab:green"), (2 + 2j, "tab:blue"), (0.3, "tab:purple")]:
    values = np.array([cmath.log(z) + 2j * math.pi * k for k in range(-1, 2)])
    ax.plot(values.real, values.imag, "o", markerfacecolor="none", color=colour, markersize=9)
    ax.plot(cmath.log(z).real, cmath.log(z).imag, "o", color=colour, markersize=9, label=f"log({z})")
ax.axhline(0, color="black", linewidth=0.8)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_yticks(np.pi * np.arange(-3, 4))
ax.set_yticklabels(["$-3\\pi$", "$-2\\pi$", "$-\\pi$", "0", "$\\pi$", "$2\\pi$", "$3\\pi$"])
ax.set_xlim(-6, 6)
ax.set_ylim(-10, 10)
ax.set_aspect("equal")
ax.set_xlabel("Re")
ax.set_ylabel("Im")
ax.set_title("The values of log z: vertical columns spaced $2\\pi$ apart")
ax.grid(True, alpha=0.3)
ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), fontsize=9)
fig.tight_layout()
plt.show()

## 6. Exercises

### Basic

1. Compute $\mathrm{Log}(-2)$, $\mathrm{Log}(3i)$ and $\mathrm{Log}(-1 - i)$.
2. List all values of $\log(-e)$ and of $\log(1 + i\sqrt3)$.
3. Compute $\mathrm{Log}(e^{4i})$. Why is the answer not $4i$?

### Intermediate

4. **Draw it first, then compute.** Sketch the image under $w = \mathrm{Log}\, z$ of the circle $|z| = 2$ traversed once counter-clockwise starting at $z = 2$. Where does the image jump, and between which two points? Then verify with a plot.
5. Show that $\mathrm{Log}(i \cdot i) = \mathrm{Log}\, i + \mathrm{Log}\, i$, but $\mathrm{Log}\big((-i)(-i)\big) \neq \mathrm{Log}(-i) + \mathrm{Log}(-i)$. Which $k$ in §2.6 occurs?
6. For which $z$ is $\mathrm{Log}(z^2) = 2\,\mathrm{Log}\, z$? Describe the region in the plane.
7. Using the branch with argument in $[0, 2\pi)$, compute $\log(-i)$, $\log(1 - i)$ and $\log(1)$. Where is this branch discontinuous?

### Advanced

8. Prove that $\mathrm{Log}(zw) - \mathrm{Log}\, z - \mathrm{Log}\, w \in \{-2\pi i, 0, 2\pi i\}$ for all nonzero $z, w$, and that it equals $0$ exactly when $\mathrm{Arg}\, z + \mathrm{Arg}\, w \in (-\pi, \pi]$.
9. Prove that there is no continuous function $L$ on the circle $|z| = 1$ with $e^{L(z)} = z$. (Hint: write $L(e^{it}) = it + 2\pi i\, k(t)$ and argue that the integer-valued $k(t)$ must be constant.)

## 7. Solutions

1. $\mathrm{Log}(-2) = \ln 2 + i\pi$. $\mathrm{Log}(3i) = \ln 3 + i\frac\pi2$. $\mathrm{Log}(-1 - i) = \ln\sqrt2 + i\,\mathrm{Arg}(-1 - i) = \frac12 \ln 2 - i\frac{3\pi}{4}$ (third quadrant, so the argument is $-3\pi/4$, not $5\pi/4$).
2. $\log(-e) = 1 + i\pi(2k + 1)$. $|1 + i\sqrt3| = 2$ and $\arg = \pi/3$, so $\log(1 + i\sqrt3) = \ln 2 + i\left(\frac\pi3 + 2\pi k\right)$.
3. $e^{4i}$ has modulus $1$ and argument $4$, which exceeds $\pi$; the principal argument is $4 - 2\pi \approx -2.283$. So $\mathrm{Log}(e^{4i}) = (4 - 2\pi)i$. $\mathrm{Log}$ only inverts $e^w$ for $\mathrm{Im}\, w \in (-\pi, \pi]$.
4. $\mathrm{Log}(2e^{it}) = \ln 2 + i\,\mathrm{Arg}(e^{it})$. For $0 \le t \le \pi$ the image climbs the vertical segment $\mathrm{Re} = \ln 2$ from $\ln 2$ up to $\ln 2 + i\pi$. Crossing $z = -2$ it jumps to $\ln 2 - i\pi$ and then climbs from there back up to $\ln 2$ at $t = 2\pi$. The image is the whole vertical segment from $\ln 2 - i\pi$ to $\ln 2 + i\pi$, traversed bottom-to-top in two pieces, with one jump of length $2\pi$.
5. $i \cdot i = -1$: $\mathrm{Log}(-1) = i\pi$ and $\mathrm{Log}\, i + \mathrm{Log}\, i = i\pi$ ✓ (sum of arguments $\pi$, which is allowed). $(-i)(-i) = -1$: $\mathrm{Log}(-1) = i\pi$ but $2\,\mathrm{Log}(-i) = -i\pi$. The difference $\mathrm{Log}(zw) - \mathrm{Log}\, z - \mathrm{Log}\, w = 2\pi i$, so $k = +1$ (the sum $-\pi$ is not in $(-\pi, \pi]$).
6. $\mathrm{Arg}(z^2) = 2\,\mathrm{Arg}\, z$ exactly when $2\,\mathrm{Arg}\, z \in (-\pi, \pi]$, i.e. $\mathrm{Arg}\, z \in (-\pi/2, \pi/2]$. The region is the open right half-plane together with the positive imaginary axis (but not the negative imaginary axis, and not $0$).
7. $\log(-i) = i\frac{3\pi}{2}$; $\log(1 - i) = \frac12 \ln 2 + i\frac{7\pi}{4}$; $\log 1 = 0$. The branch is discontinuous on the positive real axis $[0, \infty)$: just above $1$ the value is $\approx 0$, just below it is $\approx 2\pi i$.
8. Real parts: $\ln|zw| = \ln|z| + \ln|w|$, so they agree. Imaginary parts: $\mathrm{Arg}(zw)$ and $\mathrm{Arg}\, z + \mathrm{Arg}\, w$ are both arguments of $zw$, so they differ by $2\pi k$. Since $\mathrm{Arg}\, z + \mathrm{Arg}\, w \in (-2\pi, 2\pi]$ and $\mathrm{Arg}(zw) \in (-\pi, \pi]$, their difference lies in $(-3\pi, 3\pi)$, leaving $k \in \{-1, 0, 1\}$. If $\mathrm{Arg}\, z + \mathrm{Arg}\, w \in (-\pi, \pi]$ it already is the principal argument, so $k = 0$; otherwise it is not principal, so $k \neq 0$.
9. Any logarithm of $e^{it}$ has the form $it + 2\pi i\, k$ with $k$ an integer, so $k(t) = \big(L(e^{it}) - it\big)/(2\pi i)$ defines an integer-valued function on $[0, 2\pi]$. If $L$ were continuous, $k(t)$ would be continuous; a continuous integer-valued function on an interval is constant (intermediate value theorem). But $e^{i \cdot 0} = e^{i \cdot 2\pi} = 1$, so $L(1) = 0 + 2\pi i k = 2\pi i + 2\pi i k$, i.e. $0 = 2\pi i$, a contradiction.

Numerical checks:

In [ ]:
import cmath
import math

import matplotlib.pyplot as plt
import numpy as np

print("Ex 1:", cmath.log(-2), cmath.log(3j), cmath.log(-1 - 1j), " (-3pi/4 =", -3 * math.pi / 4, ")")
print("Ex 2 principal values:", cmath.log(-math.e), cmath.log(1 + 1j * math.sqrt(3)), " ln 2 =", math.log(2))
print("Ex 3:", cmath.log(cmath.exp(4j)), " 4 - 2pi =", 4 - 2 * math.pi)
print("Ex 5:", cmath.log(1j * 1j), 2 * cmath.log(1j), "|", cmath.log((-1j) * (-1j)), 2 * cmath.log(-1j))

rng = np.random.default_rng(1)
z = rng.normal(size=20_000) + 1j * rng.normal(size=20_000)
holds = np.isclose(np.log(z**2), 2 * np.log(z))
in_region = np.abs(np.angle(z)) < np.pi / 2
print("Ex 6: Log(z^2) = 2 Log z exactly in the right half-plane:", np.array_equal(holds, in_region))


def log_0_2pi(z):
    value = cmath.log(z)
    return value + 2j * math.pi if value.imag < 0 else value


print("Ex 7:", log_0_2pi(-1j), log_0_2pi(1 - 1j), log_0_2pi(1), " 7pi/4 =", 7 * math.pi / 4)

t = np.linspace(0, 2 * np.pi, 1000)
image = np.log(2 * np.exp(1j * t))
fig, ax = plt.subplots(figsize=(6, 7))
ax.plot(image.real[t <= np.pi], image.imag[t <= np.pi], "o", markersize=2, label="$0 \\leq t \\leq \\pi$")
ax.plot(image.real[t > np.pi], image.imag[t > np.pi], "o", markersize=2, label="$\\pi < t \\leq 2\\pi$")
ax.axhline(0, color="black", linewidth=0.8)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlim(-2, 3)
ax.set_aspect("equal")
ax.set_xlabel("Re")
ax.set_ylabel("Im")
ax.set_title("Exercise 4: Log of $|z| = 2$")
ax.grid(True, alpha=0.3)
ax.legend(loc="lower right")
plt.show()

## 8. Conclusion

- Every $z \neq 0$ has infinitely many logarithms $\ln|z| + i(\mathrm{Arg}\, z + 2\pi k)$: a vertical column spaced $2\pi$ apart. This is the periodicity of $e^z$ read backwards.
- The principal logarithm $\mathrm{Log}\, z = \ln|z| + i\,\mathrm{Arg}\, z$ takes values in the strip $-\pi < \mathrm{Im} \le \pi$. It is what `cmath.log`, `np.log` and sympy's `log` compute.
- $\mathrm{Log}$ jumps by $2\pi i$ across its branch cut, the negative real axis. On the cut, floating point uses the sign of a zero imaginary part to choose the side (`complex(-1, -0.0)` gives $-\pi i$); the literal `-1-0j` does **not** give a negative zero.
- No continuous logarithm exists around a loop enclosing $0$, the branch point. A different branch moves the cut but cannot remove it.
- $\mathrm{Log}(zw) = \mathrm{Log}\, z + \mathrm{Log}\, w + 2\pi i k$ with $k \in \{-1, 0, 1\}$; the plain law holds when $\mathrm{Arg}\, z + \mathrm{Arg}\, w \in (-\pi, \pi]$, and always as an identity between sets of values.
- On its Riemann surface, a spiral staircase of sheets, $\log$ is continuous and single-valued.

**Next:** [Complex powers](04_complex_powers.ipynb) defines $z^w = e^{w \log z}$. The many values of $\log$ become many values of $z^w$, which is why $i^i$ is a whole list of real numbers and why $(z^a)^b \neq z^{ab}$ in general.